In [3]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

GLOBAL_PATH = '/kaggle/input/google-research-identify-contrails-reduce-global-warming'

In [54]:
limits_spectralIndex = {(8, 9): (-2.0073962, 1.2556516),
        (8, 10): (-1.2914455, 1.9591796),
        (8, 11): (-0.65337217, 1.1425757),
        (8, 12): (-0.4293038, 0.5412206),
        (8, 13): (-0.3719291, 0.7068149),
        (8, 14): (-0.32991663, 0.5749295),
        (8, 15): (-0.2951218, 0.47614375),
        (8, 16): (-0.27196515, 0.32695884),
        (9, 10): (-2.47288, 5.0891347),
        (9, 11): (-0.7054117, 1.5090419),
        (9, 12): (-0.31053233, 0.6374393),
        (9, 13): (-0.34346375, 0.7703995),
        (9, 14): (-0.28906363, 0.6275368),
        (9, 15): (-0.24612188, 0.49756303),
        (9, 16): (-0.20720091, 0.3013589),
        (10, 11): (-1.8315057, 1.7731701),
        (10, 12): (-1.0108625, 0.67475146),
        (10, 13): (-0.73298323, 0.7571506),
        (10, 14): (-0.60375404, 0.58967346),
        (10, 15): (-0.50824136, 0.45918718),
        (10, 16): (-0.45957223, 0.3249839),
        (11, 12): (-1.5498391, 1.1068077),
        (11, 13): (-0.37653866, 0.43000376),
        (11, 14): (-0.57611704, 0.29362467),
        (11, 15): (-0.5367903, 0.19310474),
        (11, 16): (-0.4271444, 0.1274666),
        (12, 13): (-2.0401301, 3.0744278),
        (12, 14): (-0.9623841, 1.4088525),
        (12, 15): (-0.7368909, 0.82770354),
        (12, 16): (-0.525251, 0.4200794),
        (13, 14): (-1.8499428, 0.17252406),
        (13, 15): (-1.0927216, 0.20189151),
        (13, 16): (-0.72561383, 0.07278631),
        (14, 15): (-0.67102945, 0.44037148),
        (14, 16): (-0.8847657, 0.12533462),
        (15, 16): (-1.5200828, 0.3035623)}


example_record_id = '1704010292581573769'
def get_human_mask(record_id = example_record_id, **kwargs):
    # access base dir
    BASE_DIR = kwargs["BASE_DIR"]
    
    # load ground truth mask
    with open(os.path.join(BASE_DIR, record_id, 'human_pixel_masks.npy'), 'rb') as f:
        human_pixel_mask = np.load(f)
    
    return human_pixel_mask

def load_band(band_number,record_id = example_record_id, **kwargs):
    # access base dir
    BASE_DIR = kwargs["BASE_DIR"]
    
    # load band
    with open(os.path.join(BASE_DIR, record_id, f'band_{band_number:02}.npy'), 'rb') as f:
        band_i = np.load(f)
        
    return band_i

def base_spectral_index(band_numberA, band_numberB, record_id = example_record_id, limit_frame = True, **kwargs):
    # get wavelengths
    lam   = kwargs["lambda"]
    
    # individual bands
    bandA = load_band(band_numberA,record_id=record_id, **kwargs)
    bandB = load_band(band_numberB,record_id=record_id, **kwargs)

    # calculate spectral index
    spectral_index = np.log(bandA/bandB)/ np.log(lam[band_numberA]/lam[band_numberB])
    
    if limit_frame:
        # get just one frame of interest
        spectral_index = spectral_index[..., kwargs["N_TIMES_BEFORE"]]
        
    return spectral_index
def normalize_range(spectral_map, band_numberA, band_numberB, **kwargs):
    """Maps spectral_map to the range [0, 1]."""
    # get bounds
    s_min, s_max = kwargs["limits_spectralIndex"][(min(band_numberA, band_numberB),max(band_numberA, band_numberB))]
    
    # normalize
    return (spectral_map - s_min) / (s_max - s_min)

def spektral_index(band_numberA, band_numberB, record_id = example_record_id, limit_frame = False, **kwargs):
    # get spectral index map
    s_map = base_spectral_index(band_numberA, band_numberB, record_id = record_id, limit_frame = limit_frame, **kwargs)
    
    # normalize it
    s_map = normalize_range(s_map, band_numberA, band_numberB, **kwargs)
    
    # clip it
    s_map = np.clip(s_map, 0, 1)

    return s_map

def load_spektral_index(record_id = example_record_id, band_numberA = 13, band_numberB= 14, limit_frame = False, data_type = "train", **kwargs):
    # get spectral index map
    s_map = base_spectral_index(band_numberA, band_numberB, record_id = record_id, limit_frame = limit_frame, **kwargs)
    
    # normalize it
    s_map = normalize_range(s_map, band_numberA, band_numberB, **kwargs)
    
    # clip it
    s_map = np.clip(s_map, 0, 1)
    
    # get mask
    if data_type in ['train', 'validation']: 
        mask = get_human_mask(record_id = record_id, **kwargs)
    else: 
        mask = None
    
    return s_map, mask

class ContrailDataset_SI(Dataset):
    def __init__(self, base_dir, data_type='train', transform=None):
        assert data_type in ['train', 'validation', 'test'], \
            "'data_type' should be one of 'train', 'validation', or 'test'"

        self.base_dir = base_dir
        self.data_type = data_type
        self.transform = transform
        self.record = os.listdir(self.base_dir +'/'+ self.data_type)
        self.baseSettings_SpectralIndex = {
            "BASE_DIR": self.base_dir +'/'+ self.data_type,
            "example_record_id": '1704010292581573769',
            "lambda": np.array([0.,0.47,0.64,0.86,1.37,1.6,2.2,3.9,6.2,6.9,7.3,8.4,9.6,10.3,11.2,12.3,13.3]),
            "N_TIMES_BEFORE": 4,
            "limits_spectralIndex": limits_spectralIndex
        }


    def __len__(self):
        return len(self.record)

    def __getitem__(self, idx):
        record_id = self.record[idx]
        record_dir = os.path.join(self.base_dir, self.data_type, record_id)
        
        spectralIndex_map, pixel_masks = load_spektral_index(
                            record_id = record_id, 
                            band_numberA = 13, 
                            band_numberB= 14, 
                            limit_frame = False, 
                            data_type = self.data_type,
                            **self.baseSettings_SpectralIndex)
        

        # If the data type is 'train' or 'validation', load the masks
        if self.data_type not in ['train', 'validation']:
            pixel_masks = None  # No masks for 'test' data
        
        # shapes: (256,256,8) and (256,256,1)
        sample = {'frames': spectralIndex_map, 'mask': pixel_masks}

        if self.transform:
            sample = self.transform(sample)

        return sample
    
train_data = ContrailDataset_SI(GLOBAL_PATH, data_type='train')
example = train_data[0]
print(example["frames"].shape, example["mask"].shape)

(256, 256, 8) (256, 256, 1)
